# Final V1 GPU acceptance: English/Odia → Japanese/French

Use **Runtime → Change runtime type → T4 GPU**, then run every cell in order. This notebook loads each model once, uses Qwen FP32/eager/default sampling with seed 42, saves truthful artifacts, and never claims Odia acceptance until you listen to both Odia outputs.

In [ ]:
import subprocess, sys
gpu = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
assert gpu.returncode == 0, 'No NVIDIA GPU. Select a T4 runtime and reconnect.'
print(gpu.stdout)
print(sys.version)
assert sys.version_info[:2] == (3, 12), 'This notebook targets current Colab Python 3.12.'

In [ ]:
%pip install -q --upgrade "faster-whisper>=1.1,<2" "transformers>=4.57,<5" "accelerate>=1.2" "sentencepiece>=0.2" "soundfile>=0.13" "qwen-tts>=0.0.5,<0.2" "fastapi>=0.115,<1" "uvicorn[standard]>=0.30,<1" "python-multipart>=0.0.9" "psutil>=6" "nest-asyncio>=1.6"
import numpy, torch
print('NumPy', numpy.__version__, '| PyTorch', torch.__version__, '| CUDA', torch.version.cuda)
assert torch.cuda.is_available()

## Upload two separate 3–15 second WAV recordings

In [ ]:
from google.colab import files
from pathlib import Path
print('Choose the ENGLISH WAV only')
english_upload = files.upload()
assert len(english_upload) == 1
ENGLISH_WAV = Path('/content') / next(iter(english_upload))
print('Choose the ODIA WAV only')
odia_upload = files.upload()
assert len(odia_upload) == 1
ODIA_WAV = Path('/content') / next(iter(odia_upload))
assert ENGLISH_WAV.suffix.lower() == ODIA_WAV.suffix.lower() == '.wav'
print(ENGLISH_WAV, ODIA_WAV)

In [ ]:
import gc, json, os, random, time, wave, zipfile
from datetime import datetime, timezone
import numpy as np, psutil, soundfile as sf, torch
from IPython.display import Audio, display
from faster_whisper import WhisperModel
from qwen_tts import Qwen3TTSModel
from transformers import AutoProcessor, SeamlessM4Tv2Model

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
MODEL_IDS = {'asr':'small', 'translation':'facebook/seamless-m4t-v2-large', 'voice_generation':'Qwen/Qwen3-TTS-12Hz-0.6B-Base'}
def ram_mb(): return round(psutil.Process().memory_info().rss / 2**20, 1)
def vram_mb(): return round(torch.cuda.memory_allocated() / 2**20, 1)
def duration_ms(path):
    with wave.open(str(path), 'rb') as f: return round(f.getnframes()/f.getframerate()*1000, 1)
for path in (ENGLISH_WAV, ODIA_WAV):
    d = duration_ms(path)
    assert 3000 <= d <= 15000, f'{path.name} is {d/1000:.1f}s; use 3–15 seconds.'

load_started = time.perf_counter()
asr = WhisperModel('small', device='cuda', compute_type='float16')
seamless_processor = AutoProcessor.from_pretrained(MODEL_IDS['translation'])
seamless = SeamlessM4Tv2Model.from_pretrained(MODEL_IDS['translation'], torch_dtype=torch.float16, low_cpu_mem_usage=True).to('cuda').eval()
# Owner-verified T4-safe Qwen path: FP32 + eager attention. Do not add do_sample=False.
qwen = Qwen3TTSModel.from_pretrained(MODEL_IDS['voice_generation'], device_map='cuda:0', dtype=torch.float32, attn_implementation='eager')
MODEL_LOADING_MS = round((time.perf_counter()-load_started)*1000, 1)
print({'model_loading_ms': MODEL_LOADING_MS, 'ram_mb': ram_mb(), 'vram_mb': vram_mb()})

In [ ]:
TARGETS = {'ja':('jpn','Japanese'), 'fr':('fra','French')}
STAMP = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
OUT = Path('/content') / f'voice-translator-acceptance-{STAMP}'
OUT.mkdir()
PROMPTS = {}
def decode(tokens):
    row = tokens[0][0] if getattr(tokens[0], 'ndim', 1) > 1 else tokens[0]
    return seamless_processor.decode(row.detach().cpu().tolist(), skip_special_tokens=True).strip()
def run_route(source, target):
    audio_path = ENGLISH_WAV if source == 'en' else ODIA_WAV
    route_started = time.perf_counter(); asr_ms = None; transcript = None
    if source == 'en':
        t = time.perf_counter(); segments, _ = asr.transcribe(str(audio_path), language='en', beam_size=5, vad_filter=True)
        transcript = ' '.join(s.text.strip() for s in segments).strip(); asr_ms = round((time.perf_counter()-t)*1000,1)
        inputs = seamless_processor(text=transcript, src_lang='eng', return_tensors='pt').to('cuda')
        english_reference = transcript
    else:
        translation_started = time.perf_counter()
        samples, sr = sf.read(audio_path, dtype='float32')
        inputs = seamless_processor(audios=samples, sampling_rate=sr, return_tensors='pt').to('cuda')
        with torch.inference_mode(): english_reference = decode(seamless.generate(**inputs, tgt_lang='eng', generate_speech=False))
    t = translation_started if source == 'ory' else time.perf_counter()
    with torch.inference_mode(): translated = decode(seamless.generate(**inputs, tgt_lang=TARGETS[target][0], generate_speech=False))
    translation_ms = round((time.perf_counter()-t)*1000,1)
    prompt_key = source
    if prompt_key not in PROMPTS:
        t = time.perf_counter(); PROMPTS[prompt_key] = qwen.create_voice_clone_prompt(ref_audio=str(audio_path), ref_text=transcript, x_vector_only_mode=(source=='ory'))
        prompt_ms = round((time.perf_counter()-t)*1000,1)
    else: prompt_ms = 0.0
    torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
    t = time.perf_counter(); wavs, sr = qwen.generate_voice_clone(text=translated, language=TARGETS[target][1], voice_clone_prompt=PROMPTS[prompt_key])
    qwen_ms = round((time.perf_counter()-t)*1000,1)
    wav_path = OUT / f'{source}-to-{target}.wav'; sf.write(wav_path, wavs[0], sr)
    warm_ms = round((time.perf_counter()-route_started)*1000,1)
    result = {'request_id':f'{STAMP}-{source}-{target}', 'source_language':source, 'target_language':target, 'source_transcript':transcript, 'english_reference':english_reference, 'translated_text':translated, 'audio_file':wav_path.name, 'models':MODEL_IDS, 'conditioning_mode':'transcript_conditioned' if source=='en' else 'speaker_only', 'cold_start':False, 'seed':SEED, 'timings':{'model_loading_ms':0.0, 'asr_ms':asr_ms, 'translation_ms':translation_ms, 'qwen_prompt_ms':prompt_ms, 'qwen_voice_generation_ms':qwen_ms, 'inference_ms':warm_ms, 'input_duration_ms':duration_ms(audio_path), 'output_duration_ms':duration_ms(wav_path), 'real_time_factor':round(warm_ms/duration_ms(audio_path),4)}, 'process_memory_mb':ram_mb(), 'gpu_vram_mb':vram_mb(), 'acceptance':'pending_owner_listening'}
    (OUT / f'{source}-to-{target}.json').write_text(json.dumps(result, ensure_ascii=False, indent=2))
    print('\n', source, '→', target, '| English:', english_reference, '| Translation:', translated)
    display(Audio(str(audio_path))); display(Audio(str(wav_path)))
    return result
RESULTS = [run_route(s,t) for s,t in [('en','ja'),('en','fr'),('ory','ja'),('ory','fr')]]
(OUT/'session.json').write_text(json.dumps({'model_loading_ms':MODEL_LOADING_MS,'models':MODEL_IDS,'results':RESULTS}, ensure_ascii=False, indent=2))

In [ ]:
ZIP = Path('/content') / f'{OUT.name}.zip'
with zipfile.ZipFile(ZIP, 'w', zipfile.ZIP_DEFLATED) as archive:
    for path in OUT.iterdir(): archive.write(path, path.name)
print('Downloading', ZIP.name)
files.download(str(ZIP))

## Optional: temporary authenticated API for the local frontend

This reuses the already-loaded models. Run only after all four routes. The bearer token protects both translation and audio requests; the Cloudflare quick tunnel is short-lived and requires no paid account.

In [ ]:
import nest_asyncio, re, secrets, threading
from fastapi import FastAPI, File, Form, Header, HTTPException, UploadFile
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import Response
import uvicorn
TOKEN = secrets.token_urlsafe(24)
app = FastAPI(); app.add_middleware(CORSMiddleware, allow_origins=['http://localhost:5173'], allow_methods=['POST','GET'], allow_headers=['Authorization','Content-Type'])
audio_cache = {}
def auth(value):
    if value != f'Bearer {TOKEN}': raise HTTPException(401, 'Unauthorized')
@app.post('/api/translate')
async def translate_endpoint(audio: UploadFile=File(...), source_language:str=Form(...), target_language:str=Form(...), authorization:str|None=Header(None)):
    auth(authorization)
    if source_language not in {'en','ory'} or target_language not in {'ja','fr'}: raise HTTPException(422, 'Unsupported language pair')
    payload=await audio.read()
    if not payload or len(payload)>5*1024*1024: raise HTTPException(413, 'Audio must be non-empty and at most 5 MB')
    stem=secrets.token_hex(6); raw=Path('/content')/f'api-{stem}.upload'; temp=Path('/content')/f'api-{stem}.wav'; raw.write_bytes(payload)
    converted=subprocess.run(['ffmpeg','-nostdin','-v','error','-y','-i',str(raw),'-ac','1','-ar','16000','-c:a','pcm_s16le',str(temp)],capture_output=True)
    if converted.returncode or not temp.exists(): raw.unlink(missing_ok=True); raise HTTPException(422, 'Audio could not be decoded')
    if not 3000 <= duration_ms(temp) <= 15000: raw.unlink(missing_ok=True); temp.unlink(missing_ok=True); raise HTTPException(422, 'Record 3–15 seconds')
    global ENGLISH_WAV, ODIA_WAV
    old = ENGLISH_WAV if source_language=='en' else ODIA_WAV
    if source_language=='en': ENGLISH_WAV=temp
    else: ODIA_WAV=temp
    PROMPTS.pop(source_language, None)
    try: result=run_route(source_language,target_language); data=(OUT/result['audio_file']).read_bytes()
    finally:
        if source_language=='en': ENGLISH_WAV=old
        else: ODIA_WAV=old
        raw.unlink(missing_ok=True); temp.unlink(missing_ok=True)
    audio_cache[result['request_id']]=data; result['audio_url']=f"/api/audio/{result['request_id']}.wav"; return result
@app.get('/api/audio/{name}')
def audio_endpoint(name:str, authorization:str|None=Header(None)):
    auth(authorization); return Response(audio_cache[name.removesuffix('.wav')], media_type='audio/wav')
nest_asyncio.apply(); threading.Thread(target=lambda:uvicorn.run(app,host='0.0.0.0',port=8000),daemon=True).start()
subprocess.run(['wget','-q','-O','/content/cloudflared','https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64'],check=True)
os.chmod('/content/cloudflared',0o755)
tunnel=subprocess.Popen(['/content/cloudflared','tunnel','--url','http://127.0.0.1:8000'],stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
url=''
for _ in range(80):
    line=tunnel.stdout.readline(); match=re.search(r'https://[a-z0-9-]+\.trycloudflare\.com',line)
    if match: url=match.group(0); break
assert url, 'Tunnel did not start.'
print(f'VITE_API_URL={url}')
print(f'VITE_API_TOKEN={TOKEN}')